# 05 — Pro Consumidor Prices | Silver Transformation

One row per **month × product × presentation × retail segment**, price in DOP.

| Rule | Treatment |
|---|---|
| Year outside the report period | The title states the period (e.g. "Julio - Septiembre del 2026"). Rows with another year (2027–2041 in the current file) are **rejected** to `quality.dq_rejected_records`, never corrected by hand. |
| Month | Spanish month name must be valid and inside the report period. |
| Price | Numeric and > 0. |
| Label variants | The same product or presentation is sometimes written with different capitals or spacing ("Aceite de Soya Crisol" / "Aceite de soya Crisol", "Lata 106gr./ 95 g." / "Lata 106 gr./ 95 g."). Labels that are equal ignoring case and spaces get one canonical label (the most frequent one); the original label is kept in `source_product` / `source_presentation`. Labels with different words are never merged. |
| Duplicates | One row per month, product, presentation and segment; extra copies are rejected. |

Target: `silver.proconsumidor_prices_monthly` · full refresh · **Status:** Development.

In [0]:
import re, uuid, json
from datetime import datetime, timezone, date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.bronze.proconsumidor_prices_raw"
TARGET = f"{CATALOG}.silver.proconsumidor_prices_monthly"
REJECTED = f"{CATALOG}.quality.dq_rejected_records"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
MONTHS = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

b = spark.table(SOURCE)
b = b.join(b.groupBy("_source_file").agg(F.max("_ingestion_ts").alias("_ingestion_ts")), ["_source_file", "_ingestion_ts"])
rows = b.orderBy("_source_row_number").collect()
title = rows[0]["_report_title"]
m = re.search(r"([A-Za-zé]+)\s*-\s*([A-Za-zé]+)\s+del?\s+(\d{4})", title)
report_year = int(m.group(3)); m1, m2 = MONTHS[m.group(1).lower()], MONTHS[m.group(2).lower()]
print(f"Report period from title: months {m1}-{m2} of {report_year}")

good, rejects = [], []
for r in rows:
    key = f"row{r['_source_row_number']}"
    rec = {k: r[k] for k in ["id", "producto", "tipo", "precio", "mes", "anio", "establecimiento"]}
    try:
        year = int(float(r["anio"]))
    except (TypeError, ValueError):
        rejects.append(("invalid_year", key, rec, r["_batch_id"])); continue
    if year != report_year:
        rejects.append(("year_outside_report_period", key, rec, r["_batch_id"])); continue
    month = MONTHS.get(str(r["mes"]).strip().lower())
    if month is None or not (m1 <= month <= m2):
        rejects.append(("month_outside_report_period", key, rec, r["_batch_id"])); continue
    try:
        price = float(r["precio"])
    except (TypeError, ValueError):
        rejects.append(("non_numeric_price", key, rec, r["_batch_id"])); continue
    if price <= 0:
        rejects.append(("non_positive_price", key, rec, r["_batch_id"])); continue
    good.append((date(year, month, 1), " ".join(str(r["producto"]).split()), " ".join(str(r["tipo"]).split()),
                 " ".join(str(r["establecimiento"]).split()).lower(), price, r["_source_row_number"], r["_batch_id"]))

# Canonical labels: variants equal ignoring case and spaces map to their most frequent spelling
from collections import Counter, defaultdict
def canonical(values):
    groups = defaultdict(Counter)
    for v in values: groups[re.sub(r"\s+", "", v).lower()][v] += 1
    return {v: max(c.items(), key=lambda kv: (kv[1], kv[0]))[0] for c in groups.values() for v in c}
canon_prod, canon_pres = canonical([g[1] for g in good]), canonical([g[2] for g in good])
label_variants = sum(1 for g in good if canon_prod[g[1]] != g[1] or canon_pres[g[2]] != g[2])
good = [(g[0], canon_prod[g[1]], canon_pres[g[2]], g[3], g[4], g[5], g[6], g[1], g[2]) for g in good]
print(f"Rows with a non-canonical product or presentation label: {label_variants}")

df = spark.createDataFrame(good, "reference_month date, product string, presentation string, retail_segment string, price_dop double, _source_row_number int, _bronze_batch_id string, source_product string, source_presentation string")
w = Window.partitionBy("reference_month", "product", "presentation", "retail_segment").orderBy("_source_row_number")
ranked = df.withColumn("_rn", F.row_number().over(w))
dups = ranked.filter("_rn > 1").collect()
for d in dups:
    rejects.append(("duplicate_month_product_segment", f"row{d['_source_row_number']}", {k: str(d[k]) for k in ["reference_month", "product", "presentation", "retail_segment", "price_dop"]}, d["_bronze_batch_id"]))
silver = (ranked.filter("_rn = 1").drop("_rn")
          .withColumn("price_dop", F.col("price_dop").cast("decimal(12,2)"))
          .withColumn("_source_file", F.lit(rows[0]["_source_file"])).withColumn("_report_title", F.lit(title))
          .withColumn("_silver_run_id", F.lit(RUN_ID)).withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp")))

checks = []
def add(name, rtype, tested, failed):
    checks.append((RUN_ID, TARGET, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
total = len(rows); n = silver.count()
from collections import Counter
rc = Counter(x[0] for x in rejects)
for rule in ["year_outside_report_period", "month_outside_report_period", "non_numeric_price", "non_positive_price", "duplicate_month_product_segment", "invalid_year"]:
    add(rule, "enforced_reject", total, rc.get(rule, 0))
add("label_variants_canonicalized", "observational", len(good), label_variants)
add("reconciliation_silver_plus_rejected_equals_bronze", "enforced", total, abs(total - n - len(rejects)))
for c in checks: print(f"  {c[6]:<4} {c[2]:<46} failed={c[5]:,}/{c[4]:,}")
assert checks[-1][6] == "PASS"

silver.orderBy("product", "presentation", "retail_segment", "reference_month").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Silver: Pro Consumidor basic-goods prices (DOP) by month, product, presentation and retail segment. Rows with years outside the report period are rejected.'")
if rejects:
    spark.createDataFrame([(RUN_ID, TARGET, r[0], r[1], json.dumps(r[2], ensure_ascii=False), r[3], RUN_TS) for r in rejects],
        "run_id string, target_table string, rule_name string, record_key string, record_json string, source_batch_id string, detected_ts timestamp").write.mode("append").saveAsTable(REJECTED)
spark.createDataFrame([(c[0], c[1], c[2], "enforced" if c[3] != "observational" else c[3], c[4], c[5], "PASS" if c[5] == 0 else ("WARN" if c[3] == "observational" else ("REJECTED" if c[3] == "enforced_reject" else "FAIL")), c[7]) for c in checks],
    "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

s = spark.sql(f"SELECT COUNT(*) n, COUNT(DISTINCT product) p, COUNT(DISTINCT retail_segment) seg, MIN(reference_month) a, MAX(reference_month) z FROM {TARGET}").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "bronze_rows": total, "silver_rows": s["n"], "products": s["p"], "segments": s["seg"],
    "months": [str(s["a"]), str(s["z"])], "rejected": dict(rc), "label_variants_canonicalized": label_variants}, ensure_ascii=False))